# MDS — QUBO + VQE (Qiskit)

Mismo Hamiltoniano Ising que QAOA. Minimización variacional con **VQE**
(`EfficientSU2` + COBYLA) y medición en Aer.


In [1]:
import itertools
import json
import math
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np

from qiskit import transpile
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import EfficientSU2
from qiskit_aer.primitives import EstimatorV2 as AerEstimator, SamplerV2 as AerSampler
from scipy.optimize import minimize


In [2]:
nodes = ["Mario", "Sarah", "Raul", "Ana", "Enrique", "Saul"]
edges = [("Mario", "Sarah"), ("Mario", "Raul"), ("Mario", "Ana"),
         ("Enrique", "Sarah"), ("Enrique", "Raul"), ("Saul", "Ana")]
idx = {v: i for i, v in enumerate(nodes)}
n = len(nodes)
adj = {i: set() for i in range(n)}
for a, b in edges:
    adj[idx[a]].add(idx[b])
    adj[idx[b]].add(idx[a])
closed = {v: sorted(adj[v] | {v}) for v in range(n)}

def bits_cost_payments(bits):
    return int(sum(bits))

def is_dominating(bits):
    return all(any(bits[u] for u in closed[v]) for v in range(n))

def decode(bits):
    return [nodes[i] for i in range(n) if bits[i]]

# Óptimo exacto (referencia compartida)
best_cost, best_bits = n + 1, None
for k in range(2 ** n):
    bits = [(k >> i) & 1 for i in range(n)]
    if is_dominating(bits):
        c = bits_cost_payments(bits)
        if c < best_cost:
            best_cost, best_bits = c, bits
OPTIMAL_COST = float(best_cost)
OPTIMAL_SOL = decode(best_bits)
print("Óptimo MDS:", OPTIMAL_SOL, "coste =", OPTIMAL_COST)

# QUBO MDS (Lucas-style): A*sum x + B*sum (1 - sum_{N[v]} x)^2
A_PEN, B_PEN = 1.0, 2.0
const = 0.0
lin = np.zeros(n)
quad = {}
for i in range(n):
    lin[i] += A_PEN
for v in range(n):
    L = closed[v]
    const += B_PEN
    for u in L:
        lin[u] -= B_PEN
    for a in range(len(L)):
        for b in range(a + 1, len(L)):
            key = tuple(sorted((L[a], L[b])))
            quad[key] = quad.get(key, 0.0) + 2 * B_PEN

def qubo_cost(bits):
    val = const
    for i in range(n):
        val += lin[i] * bits[i]
    for (i, j), c in quad.items():
        val += c * bits[i] * bits[j]
    return float(val)

def resolve_metrics_dir() -> Path:
    for base in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        root = base / "Notebooks" / "Optimization" / "MinimumDominatingSetOptimization"
        if root.is_dir():
            d = root / "metrics"
            d.mkdir(parents=True, exist_ok=True)
            return d
    d = Path.cwd().resolve() / "Notebooks" / "Optimization" / "MinimumDominatingSetOptimization" / "metrics"
    d.mkdir(parents=True, exist_ok=True)
    return d

def save_mds_metrics(payload: dict, filename: str):
    path = resolve_metrics_dir() / filename
    path.write_text(json.dumps(payload, indent=2))
    print("Saved:", path)
    return path


Óptimo MDS: ['Ana', 'Enrique'] coste = 2.0


## Ising + VQE multi-seed


In [3]:
sparse = []
for i in range(n):
    sparse.append(("Z", [i], -lin[i] / 2))
for (i, j), c in quad.items():
    sparse += [("Z", [i], -c / 4), ("Z", [j], -c / 4), ("ZZ", [i, j], c / 4)]
H_cost = SparsePauliOp.from_sparse_list(sparse, num_qubits=n).simplify()

N_RUNS = 5
RESTARTS = 6
SHOTS = 4000
COBYLA_MAXITER = 100
ANSATZ_REPS = 1

def run_vqe(seed: int):
    t0 = time.perf_counter()
    ansatz = EfficientSU2(n, reps=ANSATZ_REPS, entanglement="linear")
    circ = transpile(ansatz, basis_gates=["ry", "rz", "cx"], optimization_level=1)
    est = AerEstimator(options={"backend_options": {"method": "statevector"}})
    n_evals = [0]
    def energy(theta):
        n_evals[0] += 1
        return float(est.run([(circ, H_cost, theta)]).result()[0].data.evs)
    rng = np.random.default_rng(seed)
    best_theta, best_e = None, np.inf
    for _ in range(RESTARTS):
        res = minimize(energy, rng.uniform(0, 2*np.pi, circ.num_parameters), method="COBYLA",
                       options={"maxiter": COBYLA_MAXITER, "rhobeg": 0.5})
        if res.fun < best_e:
            best_e, best_theta = res.fun, res.x
    meas = circ.copy(); meas.measure_all()
    counts = AerSampler(options={"backend_options": {"method": "statevector"}}).run(
        [(meas, best_theta)], shots=SHOTS
    ).result()[0].data.meas.get_counts()
    def bits_of(s):
        return [int(s[n - 1 - i]) for i in range(n)]
    best_s = min(counts, key=lambda s: qubo_cost(bits_of(s)))
    bits = bits_of(best_s)
    elapsed = time.perf_counter() - t0
    feasible = is_dominating(bits)
    pay = bits_cost_payments(bits)
    success = feasible and abs(pay - OPTIMAL_COST) < 1e-9
    return {
        "seed": seed,
        "solution": decode(bits),
        "cost": float(pay),
        "qubo_energy": float(qubo_cost(bits)),
        "feasible": feasible,
        "success": bool(success),
        "gap": float(pay - OPTIMAL_COST),
        "time_s": float(elapsed),
        "energy_evals": n_evals[0],
        "expected_energy": float(best_e),
    }

rows = [run_vqe(s) for s in range(N_RUNS)]
for r in rows:
    print(f"seed={r['seed']} success={r['success']} sol={r['solution']} cost={r['cost']} time={r['time_s']:.2f}s")

costs = np.array([r["cost"] for r in rows], dtype=float)
times = np.array([r["time_s"] for r in rows], dtype=float)
payload = {
    "problem": "mds",
    "solver": "vqe",
    "family": "quantum_inspired",
    "protocol": {"n_runs": N_RUNS, "ansatz": "EfficientSU2", "reps": ANSATZ_REPS,
                 "restarts": RESTARTS, "shots": SHOTS, "cobyla_maxiter": COBYLA_MAXITER,
                 "n_qubits": n, "backend": "aer_statevector"},
    "instance": {"nodes": nodes, "edges": [list(e) for e in edges]},
    "quality": {
        "optimal_cost": OPTIMAL_COST,
        "optimal_solution": OPTIMAL_SOL,
        "cost_mean": float(np.mean(costs)),
        "cost_best": float(np.min(costs)),
        "gap_mean": float(np.mean(costs - OPTIMAL_COST)),
    },
    "success_rate": float(np.mean([r["success"] for r in rows])),
    "time_s": {"mean": float(np.mean(times)), "std": float(np.std(times)), "total": float(np.sum(times))},
    "effort": {"energy_evals_mean": float(np.mean([r["energy_evals"] for r in rows])), "shots": SHOTS},
    "runs": rows,
}
save_mds_metrics(payload, "vqe_mds.json")


/var/folders/v8/bkfjxqf53csbmg_s6nzbyzrm0000gn/T/ipykernel_9577/3735322414.py:16: DeprecationWarning: The class ``qiskit.circuit.library.n_local.efficient_su2.EfficientSU2`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.efficient_su2 instead.
  ansatz = EfficientSU2(n, reps=ANSATZ_REPS, entanglement="linear")


seed=0 success=True sol=['Ana', 'Enrique'] cost=2.0 time=1.59s
seed=1 success=True sol=['Ana', 'Enrique'] cost=2.0 time=1.51s
seed=2 success=True sol=['Ana', 'Enrique'] cost=2.0 time=1.56s
seed=3 success=True sol=['Ana', 'Enrique'] cost=2.0 time=1.58s
seed=4 success=True sol=['Ana', 'Enrique'] cost=2.0 time=1.56s
Saved: /Users/juanprada/Documents/UNIR-MasterQC-QuantumAdvantageInDataScience/Notebooks/Optimization/MinimumDominatingSetOptimization/metrics/vqe_mds.json


PosixPath('/Users/juanprada/Documents/UNIR-MasterQC-QuantumAdvantageInDataScience/Notebooks/Optimization/MinimumDominatingSetOptimization/metrics/vqe_mds.json')

## Storytelling plots

Figuras para documento/slides → `figures/`.


In [ ]:
# --- storytelling plots (auto) ---
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

for _base in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    _lib = _base / "Notebooks" / "lib"
    if (_lib / "storytelling.py").is_file():
        _s = str(_lib)
        if _s not in sys.path:
            sys.path.insert(0, _s)
        break
else:
    raise FileNotFoundError("Notebooks/lib/storytelling.py not found")

import importlib
import storytelling as _storytelling
importlib.reload(_storytelling)
from storytelling import (
    resolve_figures_dir,
    plot_mds_instance, plot_mds_solution,
)

figures_dir = resolve_figures_dir("Optimization", "MinimumDominatingSetOptimization")
plot_mds_instance(nodes, edges, figures_dir, stem="instance")
_dom = None
if "payload" in globals() and payload.get("runs"):
    _runs = [r for r in payload["runs"] if r.get("solution") is not None]
    if _runs:
        _dom = min(_runs, key=lambda r: float(r.get("cost", 1e18))).get("solution")
if _dom is None and "sol" in globals():
    _dom = sol
if _dom is None and "OPTIMAL_SOL" in globals():
    _dom = OPTIMAL_SOL
if _dom is None and "best_b" in globals() and best_b is not None:
    _dom = decode(best_b)
if _dom is None:
    raise RuntimeError("No MDS dominating set found for storytelling plot")
plot_mds_solution(nodes, edges, _dom, figures_dir, stem="vqe", title="VQE")

plt.show()
